# **PIPELINE INTEGRADOR DASK + SPARK**

## Heart Disease Health Indicators (BRFSS 2015)

Este notebook recorre el pipeline etapa por etapa. La lógica no vive acá: vive en el paquete `clinical_bigdata_pipeline`, que es el mismo código que ejecutan los dos contenedores. El notebook importa y narra; no duplica.

```
CSV de Kaggle
     │
     ▼
DASK ──> ingesta · limpieza · variables derivadas · EDA ──> Parquet particionado
                                                              │
                                                              ▼
                                       SPARK ──> agregación · algoritmo genético · MLlib
```

> **Nota.** Las celdas de Spark necesitan PySpark y una JVM, que no están instalados en el host a propósito, para no tocar el entorno del trabajo anterior. Esa parte corre en el contenedor `pyspark` y acá se leen sus resultados.

## **EL DATASET**

253.680 registros de la encuesta telefónica BRFSS 2015 del CDC de Estados Unidos. Una fila por persona, 22 columnas numéricas, sin valores faltantes.

Dos particularidades que dan resultados silenciosamente distintos si se asumen mal:

- **`Diabetes` tiene tres niveles** (0 = no, 1 = prediabetes o gestacional, 2 = diabetes). No es binaria, a diferencia del `Diabetes_binary` del dataset que usamos en el trabajo anterior.
- **`Age` no son años**, son 13 bandas quinquenales (1 = 18-24 … 13 = 80+).

El objetivo es `HeartDiseaseorAttack`, que es la **primera** columna del archivo, con **9,42 % de positivos**. Ese desbalance justifica todo lo que viene después: un clasificador que prediga siempre "no" obtiene 90,58 % de exactitud y **recall 0**.

In [ ]:
from clinical_bigdata_pipeline import dask_stage

ruta_csv = dask_stage.ensure_dataset(dask_stage.RUTA_CSV)

print(f'Archivo     : {ruta_csv.name}')
print(f'Tamaño      : {ruta_csv.stat().st_size / 1024**2:.1f} MB')
print(f'Columnas    : {len(dask_stage.COLUMNAS)}')
print(f'Predictoras : {len(dask_stage.PREDICTORAS)}  <- largo del cromosoma del GA')

## **ETAPA #1: INGESTA Y PARTICIONAMIENTO CON DASK**

El CSV se parte en 4 fragmentos: la guía pide leer *múltiples archivos*. Partirlo es particionamiento, no amplificación — el total de filas no cambia.

Además hay que fijar `blocksize` a mano. El valor por defecto de `dd.read_csv` son 64 MiB y el archivo pesa 21,7 MiB, así que sin tocarlo Dask crearía **una sola partición** y la etapa correría secuencial. Es el tipo de bug que no falla, solo desperdicia.

In [ ]:
ddf = dask_stage.load_raw(dask_stage.RUTA_CSV)
print(f'\nParticiones: {ddf.npartitions}')

## **ETAPA #2: LIMPIEZA Y VARIABLES DERIVADAS**

La guía pide al menos dos variables derivadas. Creamos exactamente dos, y ninguna es arbitraria:

| Variable | Qué es | Rango |
|---|---|---|
| `PuntajeSaludCV` | Adaptación de *Life's Simple 7* de la American Heart Association (Lloyd-Jones et al., *Circulation* 2010). Un punto por cada componente en rango saludable. | 0-7 |
| `GrupoEtario` | Bandas etarias, y además la clave de partición del Parquet | 4 bandas |

El dataset tiene un proxy para las siete métricas de la AHA: tabaquismo, actividad física, dieta, IMC, presión, colesterol y glucemia. Esa coincidencia es lo que hace la variable defendible.

**Por qué el prefijo numérico en `GrupoEtario`** (`1_18a39`, `2_40a54`, …): Spark lee la clave de partición como *string*, y sin el prefijo el orden lexicográfico pondría `4_70ymas` antes que `1_18a39`.

La limpieza también recorta `BMI` al rango 12-60: el archivo llega hasta 98, que son autorreportes implausibles.

In [ ]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=dask_stage.N_WORKERS, threads_per_worker=1, processes=True)
cliente = Client(cluster)
cliente

In [ ]:
ddf = dask_stage.clean_and_derive(ddf)

# Hasta acá Dask no ejecutó nada: solo construyó el grafo. Los .compute() que
# hay dentro de run_eda() son lo que dispara el trabajo.
perfil, tasas = dask_stage.run_eda(ddf)
print(perfil)
tasas

El gradiente de `PuntajeSaludCV` es **monótono en los ocho niveles**, con un factor de 28× entre los extremos: de 33 % de prevalencia en quienes no cumplen ningún criterio saludable, a 1,2 % en quienes cumplen los siete.

Es el mejor predictor unidimensional que se puede construir con estas variables, y sale de combinar siete columnas que por separado no dicen tanto.

## **ETAPA #2b: SELECCION DE VARIABLES CON DASK**

Qué variables vale la pena conservar. Se calcula la correlación de cada una de las 21 predictoras con el objetivo y se toman las 10 de mayor magnitud.

Es selección tipo **filter**: mira cada variable contra el objetivo por separado, sin entrenar ningún modelo. Es mucho más barata que un *wrapper* —que evaluaría subconjuntos completos entrenando— pero tiene una limitación que conviene admitir: **no ve la redundancia entre variables**. Si dos variables miden casi lo mismo y ambas correlacionan con el objetivo, el filter se queda con las dos.

La lista que sale de acá es el **segundo traspaso Dask → Spark**, esta vez por CSV: Spark la lee para saber sobre qué variables entrenar.

In [ ]:
seleccion = dask_stage.select_features(ddf, top_n=dask_stage.TOP_VARIABLES)
seleccion

## **ETAPA #2b: ESCRITURA DEL PARQUET — EL PUNTO DE TRASPASO**

Acá termina Dask y empieza Spark. El traspaso es por **archivos Parquet en un volumen compartido**, no por objetos en memoria.

**Por qué Parquet y no CSV:**

- Es columnar: leer 3 de 24 columnas cuesta 3 columnas, no 24
- Guarda el esquema con los tipos, así que Spark no infiere nada
- Tiene estadísticas min/max por *row group*, lo que habilita *predicate pushdown*
- Se particiona por directorios, lo que habilita *partition pruning*
- Y la razón decisiva: **los dos motores lo leen de forma nativa**, que es lo que hace posible el traspaso sin escribir un serializador propio

**Por qué particionar solo por `GrupoEtario`.** Es la clave por la que Spark filtra y agrupa después, tiene cardinalidad 4 y queda balanceada. Con `Age` (13 valores) serían 104 archivos y con `Age`+`Sex` 208, de unos 15 KB cada uno — a ese tamaño abrir el *footer* de cada archivo cuesta más que leer los datos.

In [ ]:
destino = dask_stage.DIR_PARQUET
n_filas = dask_stage.write_parquet(ddf, destino)
print(f'{n_filas} filas escritas en {destino}\n')

for directorio in sorted(destino.iterdir()):
    if directorio.is_dir():
        archivos = list(directorio.glob('*.parquet'))
        peso = sum(a.stat().st_size for a in archivos) / 1024
        print(f'  {directorio.name}/  {len(archivos)} archivos, {peso:.0f} KB')

cliente.close()
cluster.close()

## **ETAPAS #3 y #4: SPARK**

A partir de acá trabaja el contenedor `pyspark`, que lee el Parquet y la lista de variables que dejó Dask:

```bash
docker compose up
```

**Etapa 3 — agregación distribuida.** Tasa de enfermedad por nivel de `PuntajeSaludCV`, con un filtro que descarta los grupos de menos de 100 personas: con una tasa base de 9,4 %, un grupo de 100 tiene unos 9 casos y un intervalo de confianza de ±6 puntos. Son ruido, no hallazgos.

**Etapa 3b — prevalencia por factor, sobre RDDs.** Para cada variable binaria y cada uno de sus valores, cuántos casos hay y sobre cuántas personas. Se hace sobre la API de RDD y no sobre DataFrames **a propósito**, porque ahí la separación entre planeación y ejecución se ve explícita:

```python
# ---------- PLANEACIÓN: transformaciones, no se ejecuta nada ----------
rdd = (sdf.select(*columnas).rdd
    .flatMap(lambda fila: [((v, int(fila[v])), (int(fila[OBJETIVO]), 1)) for v in binarias])
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])))

# ---------- EJECUCIÓN: la acción dispara el trabajo ----------
conteos = rdd.collect()
```

Hasta el `collect()` no se leyó un solo dato: `flatMap` y `reduceByKey` solo construyen el plan. Es el patrón clásico tipo *word count*, aplicado a algo que sirve.

**Etapa 4 — el modelo.** Una regresión logística de MLlib sobre las variables que eligió Dask. Cierra la narrativa: **Dask elige las variables, Spark las usa.**

Para el desbalance usamos pesos de clase, no SMOTE. MLlib no lo trae, pero además hay un argumento que no es de conveniencia: generar pacientes sintéticos a partir de registros reales es justo lo que uno evita en contexto clínico, mientras que reponderar no inventa personas.

La métrica principal es **AUC-PR**, cuya línea base es la prevalencia (0,094), no 0,5.

## **RESULTADOS**

Las celdas siguientes leen lo que dejó el contenedor de Spark. El intercambio va en los dos sentidos: Dask le pasó el Parquet a Spark, y Spark devuelve los resultados en Parquet.

In [ ]:
import pandas as pd

if (dask_stage.DIR_SALIDAS / 'metricas_modelos.csv').exists():
    display(pd.read_csv(dask_stage.DIR_SALIDAS / 'metricas_modelos.csv'))
    display(pd.read_csv(dask_stage.DIR_SALIDAS / 'tabla_comparativa.csv'))
else:
    print('Todavía no hay resultados. Ejecutá:  docker compose up')

In [ ]:
import dask.dataframe as dd

# La vuelta Spark -> Dask: lo que calculó Spark sobre RDDs se lee con Dask
ruta = dask_stage.DIR_DATOS / 'resultados' / 'prevalencias.parquet'

if ruta.exists():
    display(dd.read_parquet(ruta).compute())
else:
    print('Todavía no hay resultados de Spark.')

## **CONCLUSIÓN TÉCNICA**

### Qué herramienta para qué etapa, en un escenario real

| Etapa | Motor | Por qué |
|---|---|---|
| Ingesta, limpieza, variables derivadas | **Dask** | Es `map` sobre particiones sin shuffle, que es donde Dask es más eficiente y Spark más caro. Semántica pandas y todo en proceso Python, sin cruzar la frontera de serialización con la JVM. |
| Agregaciones y SQL | **Spark** | Catalyst optimiza el plan lógico y la API es más rica. |
| Conteos y agregaciones sobre RDDs | **Spark** | La API de RDD expone la separación entre transformaciones y acciones de forma explícita. |
| ML distribuido | **Spark MLlib** | Es una biblioteca realmente distribuida; `dask-ml` es en buena medida un envoltorio de scikit-learn. |

### Sobre el tamaño de los datos: el hallazgo honesto

253.680 filas por 22 columnas son unos **44 MB en memoria**: el 0,3 % de la RAM de la máquina donde corrimos esto. A esa escala los dos motores distribuidos son **más lentos** que pandas en las agregaciones, y arrancar la JVM cuesta del orden de 80 veces lo que cuesta hacer la agregación completa.

Eso no es un fracaso: es el resultado esperable, y es la respuesta a qué pasa si el dataset crece. Todo motor distribuido paga un **costo fijo** para comprar un **costo marginal menor**, y existe un punto de cruce. Este trabajo lo mide y muestra que 253.680 filas está por debajo de él.

### Qué se rompe primero si el dataset crece

| Escala | En memoria | Primer cuello de botella |
|---|---|---|
| ×1 | 44 MB | arranque de la JVM |
| ×10 | 440 MB | el costo fijo de Spark, todavía |
| ×100 | 4,4 GB | **RAM del driver**: el `groupby` de pandas pide 3-4× el frame y muere |
| ×1000 | 44 GB | spill del shuffle a disco, y el scheduler centralizado de Dask |

La curva de pandas no se cruza con la de Spark: **termina**. No se vuelve lenta, se muere con `MemoryError`.

### Qué resuelve Docker acá, y qué no

**Resuelve:** fijar versiones y la matriz JVM + Python + Spark, que es la fuente real de dolor en este stack; el "en mi máquina sí funciona"; y darle a quien evalúe un único comando.

**No resuelve:** el rendimiento (un contenedor no agrega núcleos); los datos (el CSV sigue teniendo que descargarse); la distribución real — dos contenedores en un portátil reparten la *topología*, no el hardware: pelean por los mismos núcleos y la misma RAM; ni la reproducibilidad de los *resultados*, porque los tiempos no son deterministas.

### Cómo llevarlo a un clúster real

Cambiar `local[*]` por la URL del master (YARN, Kubernetes o standalone) y mover las rutas del sistema de archivos local a un almacenamiento compartido (HDFS, S3 o MinIO). **Acá se cobra la decisión de diseño:** como las dos etapas se comunican por una ruta de Parquet y no por objetos en memoria, lo único que cambia es el esquema del URI — ni una línea de lógica.